# Core-satellite trend portfolio for a family office
**2270 Financial Modeling, Group Assignment** (Group: _TODO_, Members: _TODO_)

Streamlit app: _TODO URL_ · GitHub repository: https://github.com/nilscuypers18-ops/FM-Group-Project

## Setup instructions
1. Open this notebook in a fresh Google Colab session (colab.research.google.com).
2. Run all cells (Runtime > Run all). Data is downloaded live from Yahoo Finance through the `yfinance` API. No data files are needed.

The notebook is self-contained: every function used for the results is defined below. The Streamlit app uses a copy of the same functions (`strategy.py` in the repository).

In [ ]:
!pip install -q yfinance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

pd.options.display.float_format = "{:.4f}".format

## 1. Parameters and sample period
Start and end dates are set explicitly here. Prices are downloaded from February 2006 (DBC starts trading in February 2006) so that the 10-month moving average and the 12-month volatility are available before the first backtest month, January 2007.

In [ ]:
# Assets, sample period and strategy settings (dates set explicitly)
CORE = "AGG"
SATELLITE = ["SPY", "GLD", "DBC"]
TBILL = "TBILL"                       # column name we use for the cash asset
RF_TICKER = "^IRX"                    # 13-week T-bill yield, in percent p.a.

DOWNLOAD_START = "2006-02-01"         # DBC starts trading Feb 2006
BACKTEST_START = "2007-01-31"         # first month with a portfolio return
BACKTEST_END = "2026-10-31"           # last month with a portfolio return

CORE_WEIGHT = 0.60
MA_MONTHS = 10
VOL_MONTHS = 12
COST_BPS = 10

## 2. Data
* **Source:** Yahoo Finance via `yfinance`, adjusted close prices (dividends and splits reinvested), USD.
* **Assets:** AGG (US aggregate bonds), SPY (US equity), GLD (gold), DBC (commodities). Cash: 3-month US T-bill yield `^IRX`.
* **Frequency:** daily prices converted to month-end (last trading day of each month).
* **T-bill return:** monthly return in month t = `^IRX` yield at the end of month t-1 / 100 / 12.

In [ ]:
# --- Data functions ---
def download_data(start=DOWNLOAD_START, end=BACKTEST_END):
    """Download daily adjusted closes from Yahoo Finance and convert to
    month-end prices. Returns (prices, rf):
      prices : DataFrame of month-end adjusted close prices (USD)
      rf     : Series of monthly T-bill returns (decimal)
    """
    import yfinance as yf

    tickers = [CORE] + SATELLITE + [RF_TICKER]
    # yfinance's `end` is exclusive, so add one day to include the end date
    end_excl = (pd.Timestamp(end) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
    raw = yf.download(tickers, start=start, end=end_excl,
                      auto_adjust=True, progress=False)["Close"]
    monthly = raw.resample("ME").last()      # last trading day of each month

    prices = monthly[[CORE] + SATELLITE].dropna()
    rf = tbill_monthly_return(monthly[RF_TICKER]).reindex(prices.index)
    return prices, rf


def tbill_monthly_return(irx_yield):
    """Convert the ^IRX yield (percent per year) into a monthly return.
    The yield observed at the end of month t-1 is what an investor earns
    during month t, so we lag it by one month."""
    return (irx_yield / 100 / 12).shift(1)


def asset_returns(prices, rf):
    """Simple monthly returns of all assets, with T-bills as an extra column."""
    rets = prices.pct_change()
    rets[TBILL] = rf
    return rets

In [ ]:
prices, rf = download_data(DOWNLOAD_START, BACKTEST_END)
print(prices.index[0].date(), "to", prices.index[-1].date(), "|", len(prices), "month-ends")
prices.tail()

## 3. Signals and weights
At each month-end t:
1. An asset is **on** if its price is above its 10-month moving average.
2. The 40% satellite budget is split across SPY, GLD and DBC in proportion to 1 / (12-month volatility).
3. The share of an asset that is **off** goes to T-bills. AGG stays at 60%.

These weights are held during month t+1, so every decision uses only information available at the time.

In [ ]:
# --- Signal functions ---
def trend_signal(prices, ma_months=MA_MONTHS):
    """1 if month-end price > its moving average over the last `ma_months`
    month-ends (including the current one), else 0."""
    ma = prices.rolling(ma_months).mean()
    signal = (prices > ma).astype(float)
    return signal.where(ma.notna())          # NaN until the MA is available


def rolling_vol(prices, vol_months=VOL_MONTHS):
    """Annualized volatility of monthly returns over the last `vol_months`.
    At least 10 returns are required (DBC data only starts in Feb 2006)."""
    rets = prices.pct_change()
    return rets.rolling(vol_months, min_periods=10).std() * np.sqrt(12)

In [ ]:
# --- Weight functions ---
def strategy_weights(prices, core_weight=CORE_WEIGHT, ma_months=MA_MONTHS,
                     vol_months=VOL_MONTHS):
    """Target weights decided at each month-end t (to be held during t+1)."""
    sat = prices[SATELLITE]
    signal = trend_signal(sat, ma_months)
    inv_vol = 1 / rolling_vol(sat, vol_months)

    # Each satellite asset's share of the satellite budget (sums to 1)
    share = inv_vol.div(inv_vol.sum(axis=1), axis=0)
    sat_budget = 1 - core_weight

    w = pd.DataFrame(index=prices.index)
    w[CORE] = core_weight
    for a in SATELLITE:
        w[a] = sat_budget * share[a] * signal[a]   # 0 if the asset is "off"
    w[TBILL] = 1 - w[[CORE] + SATELLITE].sum(axis=1)  # the rest sits in T-bills
    w[TBILL] = w[TBILL].clip(lower=0).round(12)       # remove tiny rounding noise

    # Only keep months where every input was available
    ready = signal.notna().all(axis=1) & inv_vol.notna().all(axis=1)
    return w[ready]


def fixed_weights(index, weights):
    """Constant target weights, e.g. {'SPY': 0.6, 'AGG': 0.4}."""
    return pd.DataFrame(weights, index=index)

In [ ]:
weights = strategy_weights(prices, CORE_WEIGHT, MA_MONTHS)
weights.loc[BACKTEST_START:].plot.area(figsize=(10, 4), title="Strategy weights (decided at month-end, held next month)")
plt.ylabel("Weight"); plt.margins(0, 0); plt.show()
weights.tail()

## 4. Backtest
Comparison portfolios:
* **60/40 SPY/AGG** benchmark, rebalanced monthly.
* **Buy & hold (same assets):** 60% AGG and 13.3% each in SPY, GLD and DBC, bought once in January 2007 and never rebalanced.

All portfolios pay 10 bps per unit of turnover, including the initial purchase.

In [ ]:
# --- Backtest functions ---
def backtest(target_weights, rets, start=BACKTEST_START, end=BACKTEST_END,
             cost_bps=COST_BPS, rebalance=True):
    """Monthly backtest.

    target_weights : weights decided at month-end t (rows) per asset (cols)
    rets           : monthly asset returns (incl. TBILL)
    rebalance      : True  -> trade back to the target every month
                     False -> buy once at the start, then let weights drift
                              (buy & hold)

    Timing: weights chosen at end of month t earn the returns of month t+1.
    Cost: cost_bps / 10,000 * turnover, deducted in the month of the trade.
    Turnover = sum over assets of |new weight - weight before trading|.
    The initial purchase (from cash) counts as turnover of 1.

    Returns a DataFrame with columns: gross, cost, net, turnover.
    """
    cols = target_weights.columns
    rets = rets[cols]
    months = rets.loc[start:end].index
    cost_rate = cost_bps / 10_000

    out = []
    held = pd.Series(0.0, index=cols)          # start from nothing
    for i, month in enumerate(months):
        decision_date = rets.index[rets.index.get_loc(month) - 1]   # end of t
        if rebalance or i == 0:
            new_w = target_weights.loc[decision_date]
        else:
            new_w = held
        turnover = (new_w - held).abs().sum()
        r = rets.loc[month].fillna(0.0)
        gross = (new_w * r).sum()
        cost = turnover * cost_rate
        out.append({"date": month, "gross": gross, "cost": cost,
                    "net": gross - cost, "turnover": turnover})
        # weights drift with returns until the next rebalance
        held = new_w * (1 + r) / (1 + gross)

    return pd.DataFrame(out).set_index("date")


def run_all(prices, rf, core_weight=CORE_WEIGHT, ma_months=MA_MONTHS,
            cost_bps=COST_BPS, start=BACKTEST_START, end=BACKTEST_END):
    """Run the strategy and the comparison portfolios.
    Returns (net monthly returns DataFrame, strategy weights, strategy result)."""
    rets = asset_returns(prices, rf)

    w_strat = strategy_weights(prices, core_weight, ma_months)

    # Same assets without the trend filter: core + equal satellite split
    sat_each = (1 - core_weight) / len(SATELLITE)
    static = {CORE: core_weight, **{a: sat_each for a in SATELLITE}}
    w_static = fixed_weights(rets.index, static)

    w_6040 = fixed_weights(rets.index, {"SPY": 0.60, "AGG": 0.40})

    res_strat = backtest(w_strat, rets, start, end, cost_bps)
    results = pd.DataFrame({
        "Strategy": res_strat["net"],
        "60/40 SPY/AGG": backtest(w_6040, rets, start, end, cost_bps)["net"],
        "Buy & hold (same assets)": backtest(w_static, rets, start, end,
                                             cost_bps, rebalance=False)["net"],
    })
    rf_bt = rets[TBILL].loc[results.index]
    return results, rf_bt, w_strat.loc[: results.index[-1]], res_strat

In [ ]:
# --- Performance functions ---
def cumulative(returns):
    """Growth of 1 unit of capital, starting at 1 before the first month."""
    growth = (1 + returns).cumprod()
    first = growth.index[0] - pd.offsets.MonthEnd(1)
    start_row = pd.DataFrame(1.0, index=[first], columns=growth.columns) \
        if isinstance(growth, pd.DataFrame) else pd.Series(1.0, index=[first])
    return pd.concat([start_row, growth])


def max_drawdown(returns):
    growth = (1 + returns).cumprod()
    return (growth / growth.cummax() - 1).min()


def performance_table(results, rf, benchmark="60/40 SPY/AGG"):
    """Performance measures from monthly returns.

    Assumptions:
      * Annualized average return = 12 x mean monthly return (arithmetic).
      * Volatility = sqrt(12) x std of monthly returns.
      * Sharpe = annualized mean excess return over T-bills / annualized
        std of excess returns.
      * Beta and alpha from an OLS regression of monthly excess returns of
        the portfolio on monthly excess returns of the benchmark.
        Beta is NOT annualized; alpha = 12 x monthly intercept.
      * Information ratio = annualized mean active return / annualized
        tracking error (active return = portfolio - benchmark).
    """
    bench = results[benchmark]
    rows = {}
    for name in results.columns:
        r = results[name]
        ex = r - rf
        ex_b = bench - rf
        beta, alpha_m = np.polyfit(ex_b, ex, 1)
        active = r - bench
        te = active.std() * np.sqrt(12)
        rows[name] = {
            "Ann. return": r.mean() * 12,
            "CAGR": (1 + r).prod() ** (12 / len(r)) - 1,
            "Volatility": r.std() * np.sqrt(12),
            "Sharpe": ex.mean() * 12 / (ex.std() * np.sqrt(12)),
            "Alpha (ann.)": alpha_m * 12,
            "Beta": beta,
            "Tracking error": te,
            "Information ratio": active.mean() * 12 / te if te > 1e-12 else np.nan,
            "Max drawdown": max_drawdown(r),
        }
    return pd.DataFrame(rows).T

In [ ]:
results, rf_bt, w_strat, res_strat = run_all(prices, rf, CORE_WEIGHT, MA_MONTHS, COST_BPS,
                                              BACKTEST_START, BACKTEST_END)
cum = cumulative(results)
cum.plot(figsize=(10, 5), title="Cumulative performance (start = 1, net of costs)")
plt.ylabel("Value of 1 invested"); plt.grid(alpha=0.3); plt.show()

## 5. Performance measures
Calculation assumptions:
* Monthly returns, net of transaction costs. Risk-free rate = 3-month T-bill.
* **Annualized average return** = 12 × mean monthly return. CAGR is shown as well.
* **Volatility** = √12 × standard deviation of monthly returns.
* **Sharpe ratio** = annualized mean excess return over T-bills / annualized volatility of excess returns.
* **Beta and alpha:** OLS regression of the portfolio's monthly excess return on the 60/40 benchmark's monthly excess return. Beta is **not** annualized; alpha = 12 × monthly intercept.
* **Information ratio** = annualized mean active return (portfolio − benchmark) / annualized tracking error.

In [ ]:
table = performance_table(results, rf_bt)
table.style.format({c: "{:.2%}" for c in ["Ann. return", "CAGR", "Volatility", "Alpha (ann.)",
                                         "Tracking error", "Max drawdown"]}
                   | {c: "{:.2f}" for c in ["Sharpe", "Beta", "Information ratio"]})

## 6. Implementation: turnover and costs

In [ ]:
print(f"Average monthly turnover (excl. first month): {res_strat['turnover'].iloc[1:].mean():.2%}")
print(f"Total costs paid (sum of monthly cost drag): {res_strat['cost'].sum():.2%}")
print(f"Share of months with each asset 'on':")
print((w_strat.loc[BACKTEST_START:, SATELLITE] > 0).mean().round(2))